### Downloaded data to text file "Shakespeare.txt"
### Combine all text into a single string

In [8]:
# Load text file (replace path with your file)
file_path = "shakespeare.txt"

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

words = text.split()
print("Number of words:", len(words))

print("Length of text:", len(text))
print(text[:500])  # preview


Number of words: 17589
Length of text: 94275
THE SONNETS

by William Shakespeare

From fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
Making a famine where abundance lies,
Thy self thy foe, to thy sweet self too cruel:
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
Within thine own bud buriest


### Create character-level vocabulary. 
#### Encode text as integer sequence. Create sequences of length 50 for training

In [9]:
import numpy as np

# Create vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Vocabulary size:", vocab_size)

# Create mappings
char_to_idx = {c:i for i,c in enumerate(chars)}
idx_to_char = {i:c for i,c in enumerate(chars)}

# Encode entire text
encoded_text = np.array([char_to_idx[c] for c in text])

# Create sequences of length 50
seq_length = 50

X = []
y = []

for i in range(len(encoded_text) - seq_length):
    X.append(encoded_text[i:i+seq_length])
    y.append(encoded_text[i+seq_length])

X = np.array(X)
y = np.array(y)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Vocabulary size: 61
Input shape: (94225, 50)
Target shape: (94225,)


### Implement a 2-layer LSTM model (hidden size = 256). 
### Display LSTM architecture and number of trainable parameters.

In [11]:
%pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout

model = Sequential([
    Embedding(vocab_size, 128, input_length=seq_length),

    LSTM(256, return_sequences=True),
    Dropout(0.2),

    LSTM(256),
    Dropout(0.2),

    Dense(vocab_size, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Display architecture
model.summary()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\rramal4\myRepos\MTech-Labs\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Train model for 20 epochs. Generate 100 characters of text given seed string "Once upon" and display the generated Shakespeare-style text.

In [ ]:
#Train model for 20 epochs; batch size 128
history = model.fit(X, y,
                   epochs=20,
                   batch_size=256)

Epoch 1/20
 40/369 ━━━━━━━━━━━━━━━━━━━━ 5:08 938ms/step - accuracy: 0.1353 - loss: 3.5188

### Generate 100 characters of text given seed string "Once upon" and display the generated Shakespeare-style text

In [10]:

import socket
import datetime
# Generate text using the trained model; 
def generate_text(seed_text, length=100, temperature=0.8):
    input_seq = [char_to_idx[c] for c in seed_text]

    for _ in range(length):
        padded = np.array(input_seq[-seq_length:])
        
        if len(padded) < seq_length:
            padded = np.pad(padded, (seq_length-len(padded), 0))

        padded = np.reshape(padded, (1, seq_length))

        preds = model.predict(padded, verbose=0)[0]

        # Temperature sampling
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))

        next_index = np.random.choice(len(preds), p=preds)
        input_seq.append(next_index)

    return seed_text + ''.join(idx_to_char[i] for i in input_seq[len(seed_text):])


timestamp = datetime.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z")
machine_id = socket.gethostname()
identifier=62
print("=" * 88)
print("NLP Assignment 1: LSTM Language Model for Text Generation")
print(f"Identifier   : {identifier}")
print(f"Timestamp    : {timestamp}")
print(f"Machine ID   : {machine_id}")
print("=" * 88)

output = generate_text("Once upon", 100)
print("\nGenerated Text:\n")
print(output)

NLP Assignment 1: LSTM Language Model for Text Generation
Identifier   : 62
Timestamp    : 2026-06-11 21:52:08 India Standard Time
Machine ID   : LHTU05CG30853L9


NameError: name 'model' is not defined

## Comparison: N‑Gram vs RNN/LSTM Language Models (Compact Summary)
### N‑Gram Models

Predict next token using last n‑1 tokens (fixed window)
Simple and interpretable
Limited by short context
Suffer from data sparsity and poor generalization

### RNN Models

Use sequential processing + hidden state to capture context
Handle variable-length dependencies better than N‑Grams
Struggle with long-term dependencies due to vanishing gradients

### LSTM Models

Enhanced RNNs with memory cell + gates (input, forget, output)
Effectively capture long-range dependencies
Learn what to keep, update, or discard over time

### Key Difference

N‑Gram: fixed, limited context
LSTM: dynamic, learned memory of long sequences

### Conclusion

N‑Grams: suitable for simple, short-context tasks
LSTMs: better for complex language tasks requiring long-term context and richer patterns